# 2.2 · Statelessness and server/discover

**无状态协议与 server/discover**

> **After this lesson you can:** Explain why 2026-07-28 MCP is stateless, design cross-call state with server-minted handles, and use server/discover and per-request version negotiation correctly.

Official reading: [Spec: Discovery](https://modelcontextprotocol.io/specification/2026-07-28/server/discover) · [Spec: Statelessness](https://modelcontextprotocol.io/specification/2026-07-28/basic/index#statelessness) · [Spec: Version negotiation](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. Call `server/discover` by hand
We start a small shop server as a subprocess and send `server/discover` as a raw JSON line. The server declares cache hints for discovery, so you can see `ttlMs` and `cacheScope` set by the server.

In [ ]:
import itertools
import json
import os
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
from pathlib import Path


def show(obj, limit=1200):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text if len(text) <= limit else text[:limit] + "\n  ...")


workdir = Path(tempfile.mkdtemp())
(workdir / "shop_server.py").write_text(textwrap.dedent('''
    import json, os, secrets, sqlite3, sys, time
    from mcp.server import CacheHint, MCPServer
    from mcp.server.mcpserver.exceptions import ToolError

    INSTANCE = sys.argv[1]                       # which replica am I?
    DB = os.environ["SHOP_DB"]                   # storage shared by every replica
    mcp = MCPServer(f"shop-{INSTANCE}", version="1.0.0",
                    instructions="Create a basket with create_basket, then pass its basket_id to add_item.",
                    cache_hints={"server/discover": CacheHint(ttl_ms=3_600_000, scope="public")})

    _implicit_cart = []                          # per-process memory: the anti-pattern

    def db():
        conn = sqlite3.connect(DB)
        conn.execute("create table if not exists baskets (id text primary key, items text, expires real)")
        return conn

    @mcp.tool()
    def add_to_my_cart(sku: str) -> str:
        """ANTI-PATTERN: add to 'the' cart remembered by this process."""
        _implicit_cart.append(sku)
        return json.dumps({"served_by": INSTANCE, "cart": _implicit_cart})

    @mcp.tool()
    def create_basket() -> str:
        """Create an empty basket and return its basket_id. Baskets expire after 10 minutes of inactivity."""
        basket_id = "bsk_" + secrets.token_urlsafe(12)      # opaque, high entropy
        with db() as conn:
            conn.execute("insert into baskets values (?, ?, ?)", (basket_id, "[]", time.time() + 600))
        return json.dumps({"served_by": INSTANCE, "basket_id": basket_id})

    @mcp.tool()
    def add_item(basket_id: str, sku: str) -> str:
        """Add a product to a basket created by create_basket."""
        with db() as conn:
            row = conn.execute("select items, expires from baskets where id = ?", (basket_id,)).fetchone()
            if row is None or row[1] < time.time():
                raise ToolError(f"Unknown or expired basket {basket_id!r}. Call create_basket to start a new one.")
            items = json.loads(row[0]) + [sku]
            conn.execute("update baskets set items = ?, expires = ? where id = ?",
                         (json.dumps(items), time.time() + 600, basket_id))
        return json.dumps({"served_by": INSTANCE, "basket_id": basket_id, "items": items})

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))

ENV = {**os.environ, "SHOP_DB": str(workdir / "shop.sqlite")}
ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "notebook", "version": "0.1"}}


class Replica:
    """One server process we talk to with raw JSON lines."""

    def __init__(self, name):
        self.name = name
        self.proc = subprocess.Popen([sys.executable, str(workdir / "shop_server.py"), name], env=ENV,
                                     stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                                     text=True)
        self.inbox = queue.Queue()
        threading.Thread(target=lambda: [self.inbox.put(l) for l in self.proc.stdout], daemon=True).start()

    def send(self, message: dict, wait=10):
        self.proc.stdin.write(json.dumps(message) + "\n")
        self.proc.stdin.flush()
        return json.loads(self.inbox.get(timeout=wait))

    def rpc(self, id_, method, params=None, meta=ENVELOPE):
        return self.send({"jsonrpc": "2.0", "id": id_, "method": method, "params": {**(params or {}), "_meta": meta}})

    def close(self):
        self.proc.stdin.close()
        return self.proc.wait(timeout=10)


a = Replica("A")
discover = a.rpc("discover-1", "server/discover")
show(discover)

Pull the `DiscoverResult` apart the way a host would:

In [ ]:
d = discover["result"]
print("server          :", d["_meta"]["io.modelcontextprotocol/serverInfo"])
print("versions        :", d["supportedVersions"])
print("capabilities    :", sorted(d["capabilities"]))
print("instructions    :", d.get("instructions"))
print(f"cache           : fresh for {d['ttlMs'] / 1000:.0f}s, scope={d['cacheScope']!r} "
      "(public = a shared gateway may cache it)")

### 2. Per-request version negotiation
No handshake: we call a tool directly at a version the server does not know, read `-32022`, choose a mutual version from `data.supported`, and retry. This is the client logic the spec describes, in about ten lines.

In [ ]:
CLIENT_VERSIONS = ["2099-01-01", "2026-07-28"]          # newest first


def call_with_negotiation(replica, method, params, preferred=CLIENT_VERSIONS[0]):
    version, attempt = preferred, 0
    while True:
        attempt += 1
        reply = replica.rpc(100 + attempt, method, params,
                            meta={**ENVELOPE, "io.modelcontextprotocol/protocolVersion": version})
        if "error" in reply and reply["error"]["code"] == -32022:
            supported = reply["error"]["data"]["supported"]
            print(f"attempt {attempt}: {version} rejected; server supports {supported}")
            mutual = [v for v in CLIENT_VERSIONS if v in supported]
            if not mutual:
                raise RuntimeError("no mutually supported protocol version")
            version = mutual[0]
            continue
        print(f"attempt {attempt}: accepted at {version}")
        return reply


reply = call_with_negotiation(a, "tools/call", {"name": "create_basket", "arguments": {}})
print(reply["result"]["content"][0]["text"])

### 3. Any replica can answer: two independent processes
Start a second process, `B`. It shares nothing with `A` except the database file the *application* chose to share. A round-robin "load balancer" sends each request to the next replica.

In [ ]:
b = Replica("B")
balancer = itertools.cycle([a, b])
ids = itertools.count(1000)


def lb_call(tool, arguments):
    replica = next(balancer)
    reply = replica.rpc(next(ids), "tools/call", {"name": tool, "arguments": arguments})
    result = reply["result"]
    return replica.name, result["isError"], result["content"][0]["text"]


for _ in range(2):
    replica = next(balancer)
    info = replica.rpc(next(ids), "server/discover")["result"]["_meta"]["io.modelcontextprotocol/serverInfo"]
    print(f"discover via {replica.name}: {info}")

### 4. Break it: implicit per-process state
`add_to_my_cart` keeps "the" cart in process memory, the way a session-era server might have kept it in a session. Behind a load balancer, the user's cart is split across replicas:

In [ ]:
for sku in ["apple", "pear", "fig", "kiwi"]:
    name, is_error, text = lb_call("add_to_my_cart", {"sku": sku})
    print(f"via {name}: {text}")

Neither replica has the full cart, and nothing in the protocol can fix that. The server inferred continuity from the process, which the spec forbids.

### 5. Fix it: a server-minted handle passed as a tool argument
Now the stateful workflow the way the spec recommends. `create_basket` mints an opaque id, and every later call names it. Whichever replica serves a call, it finds the basket in shared storage.

In [ ]:
name, _, text = lb_call("create_basket", {})
basket_id = json.loads(text)["basket_id"]
print(f"created by {name}: {basket_id}")

for sku in ["apple", "pear", "fig", "kiwi"]:
    name, is_error, text = lb_call("add_item", {"basket_id": basket_id, "sku": sku})
    print(f"via {name}: {json.loads(text)['items']}")

name, is_error, text = lb_call("add_item", {"basket_id": "bsk_does_not_exist", "sku": "x"})
print(f"\nunknown handle via {name}: isError={is_error} -> {text}")

The unknown-handle case is a **tool execution error** (`isError: true`) with a message that tells the model how to recover, exactly what the spec's "expiry errors" guidance asks for. (In this SDK, raise `ToolError` to send your own message. Any other exception is reported as a generic "Error executing tool …".)

### 6. Capabilities are judged per request, even on one connection
Two requests on the **same** process, one declaring `elicitation` and one not. The server must not carry the first declaration over to the second.

In [ ]:
(workdir / "confirm_server.py").write_text(textwrap.dedent('''
    from typing import Annotated
    from pydantic import BaseModel
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Elicit, Resolve

    mcp = MCPServer("confirm")

    class Confirm(BaseModel):
        ok: bool

    def ask(basket_id: str):
        return Elicit(f"Empty basket {basket_id}?", Confirm)

    @mcp.tool()
    def empty_basket(basket_id: str, confirm: Annotated[Confirm, Resolve(ask)]) -> str:
        """Empty a basket after the user confirms."""
        return "emptied" if confirm.ok else "kept"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
confirm = Replica.__new__(Replica)
confirm.name = "confirm"
confirm.proc = subprocess.Popen([sys.executable, str(workdir / "confirm_server.py")], stdin=subprocess.PIPE,
                                stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
confirm.inbox = queue.Queue()
threading.Thread(target=lambda: [confirm.inbox.put(l) for l in confirm.proc.stdout], daemon=True).start()

args = {"name": "empty_basket", "arguments": {"basket_id": basket_id}}
with_cap = confirm.rpc(1, "tools/call", args, meta={**ENVELOPE, "io.modelcontextprotocol/clientCapabilities": {"elicitation": {}}})
without = confirm.rpc(2, "tools/call", args)
print("request 1 (declares elicitation):", with_cap["result"]["resultType"])
print("request 2 (declares nothing)    :", without["error"]["code"], without["error"]["message"][:60])
print("                                  ", without["error"]["data"])

### 7. The legacy era: `initialize` on a fresh process
This SDK is **dual-era**. If the first request a stdio process sees is a 2025-style `initialize`, the whole process is served with legacy semantics, and a modern request on it is refused. That is why a dual-era *client* probes with `server/discover` first.

In [ ]:
old = Replica("legacy")
show(old.send({"jsonrpc": "2.0", "id": 1, "method": "initialize",
               "params": {"protocolVersion": "2025-11-25", "capabilities": {},
                          "clientInfo": {"name": "old-client", "version": "1.0"}}}), limit=600)
old.proc.stdin.write(json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"}) + "\n")
old.proc.stdin.flush()
show(old.rpc(2, "server/discover"))

### 8. The SDK does all of this for you
`Client` in its default `mode="auto"` probes `server/discover`, adopts the result, and stamps the envelope on every later request. `mode="legacy"` forces the old handshake. Compare:

In [ ]:
from mcp import Client, StdioServerParameters

params = StdioServerParameters(command=sys.executable, args=[str(workdir / "shop_server.py"), "C"], env=ENV)
async with Client(params) as client:
    d = client.session.discover_result
    print("auto  :", client.protocol_version, "| discover:", d.supported_versions, d.ttl_ms, d.cache_scope)
    print("        instructions:", client.instructions)
async with Client(params, mode="legacy") as client:
    print("legacy:", client.protocol_version, "| discover result:", client.session.discover_result)

for replica in (a, b, confirm, old):
    replica.close()

### Your turn
1. Add a `view_basket(basket_id)` tool and confirm that a basket created through replica `A` can be viewed through replica `B`. Then delete `shop.sqlite` between calls: what should the tool return, and does yours?
2. Make `create_basket` mint ids like `bsk_1`, `bsk_2`, … and explain, with the spec's four handle concerns, two concrete attacks that this enables on an unauthenticated server.
3. Change the discover cache hint to `scope="private"`. Who may still cache the result, and when would you need `private` for discovery?
4. Write `probe_era(replica)` that implements the stdio backward-compatibility rules from the lesson (discover → modern; recognised modern error → modern; anything else or a timeout → legacy). Test it on a fresh replica, and on another fresh replica that you first open with `initialize` as in section 7.